# Moda Lane Boutique
## Business Intelligence, Customer Analytics & Marketing Recommendations

**Client:** Moda Lane Boutique  
**Project:** Career180 – Corporate  
**Scope:** Data quality, exploratory analysis, sales and marketing performance, customer-type analysis, segmentation, loyalty opportunities, upselling proxies, and actionable business recommendations.

> **Data integrity note:** This dataset contains 2,000 campaign/performance records and 8 columns. It does not contain customer IDs, transaction/order IDs, product IDs, product quantities, or item-level basket data. Therefore, true customer-level RFM, individual Customer Lifetime Value (CLV), cohort retention, repeat-purchase intervals, and product-affinity analysis cannot be calculated. This notebook explicitly separates measurable KPIs from unavailable analyses and does not fabricate customer-level results.

### Business questions
1. Which marketing channels and service types generate the strongest revenue and return?
2. How do new and returning customer records contribute to conversions and revenue?
3. How does performance vary over time and across seasons?
4. Where is marketing spend potentially inefficient?
5. What loyalty, upselling, and budget-allocation actions are supported by the available evidence?

## 1. Introduction — Business Problem & Objectives

Moda Lane Boutique wants to use historical performance data to improve marketing efficiency, understand customer-type contribution, and identify opportunities for loyalty and upselling.

**Objectives**
- Establish a reproducible data-quality and cleaning workflow.
- Quantify revenue, advertising spend, conversions, ROAS, CPA, and conversion efficiency.
- Compare channels, service types, customer types, seasons, and time periods.
- Identify high-performing and underperforming combinations.
- Create evidence-based, rule-based business recommendations.
- Document data limitations and the additional fields required for advanced customer analytics.

## 2. Data Loading — Libraries and Source File

In [ ]:
# If required, install dependencies in your environment:
# %pip install pandas numpy matplotlib seaborn plotly scipy scikit-learn openpyxl

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette="deep")

# Place the CSV in the same folder as this notebook, or update DATA_PATH.
possible_paths = [
    Path("Data - Sheet1(1).csv"),
    Path("Data - Sheet1.csv"),
    Path("data.csv"),
    Path("/mnt/data/Data - Sheet1(1).csv"),
]
DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Place 'Data - Sheet1(1).csv' beside this notebook "
        "or update DATA_PATH to the CSV location."
    )

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())

## 3. Data Understanding — Structure, Columns & Initial Profiling

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\\nInfo:")
df.info()
print("\\nDescriptive statistics:")
display(df.describe(include="all").T)

In [ ]:
# Normalize column labels without changing their meaning.
df.columns = [str(c).strip() for c in df.columns]

expected_columns = [
    "Date", "Season", "Channel", "Service Type",
    "Customer Type", "Ad Spend", "Conversions", "Revenue"
]
missing_columns = sorted(set(expected_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

print("Unique values by categorical column:")
for col in ["Season", "Channel", "Service Type", "Customer Type"]:
    print(f"\\n{col} ({df[col].nunique(dropna=False)} unique)")
    display(df[col].value_counts(dropna=False).to_frame("records"))

## 4. Data Quality — Missing Values, Duplicates & Inconsistencies

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True),
})
display(quality)

duplicate_count = int(df.duplicated().sum())
print(f"Exact duplicate rows: {duplicate_count:,}")

# Check numeric values and category whitespace.
numeric_cols = ["Ad Spend", "Conversions", "Revenue"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["Season", "Channel", "Service Type", "Customer Type"]:
    df[col] = df[col].astype("string").str.strip()

print("\\nNegative-value checks:")
display(pd.DataFrame({
    "negative_count": {c: int((df[c] < 0).sum()) for c in numeric_cols},
    "zero_count": {c: int((df[c] == 0).sum()) for c in numeric_cols},
}))

In [ ]:
# Date parsing diagnostics.
parsed_dates = pd.to_datetime(df["Date"], errors="coerce")
print("Unparseable dates:", int(parsed_dates.isna().sum()))

# Potential logical checks (not automatically treated as errors):
print("Rows with zero conversions:", int((df["Conversions"] == 0).sum()))
print("Rows with zero revenue:", int((df["Revenue"] == 0).sum()))
print("Rows with conversions > 0 but zero revenue:",
      int(((df["Conversions"] > 0) & (df["Revenue"] == 0)).sum()))
print("Rows with zero ad spend:", int((df["Ad Spend"] == 0).sum()))

## 5. Data Cleaning — Reproducible Preparation

In [ ]:
clean_df = df.copy()

# Parse dates and enforce numeric types.
clean_df["Date"] = pd.to_datetime(clean_df["Date"], errors="coerce")
for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

# Remove exact duplicates only; do not remove repeated dates because
# multiple records may legitimately belong to the same date.
before = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
duplicates_removed = before - len(clean_df)

# Drop rows missing essential fields. No imputation is applied to core
# business metrics because invented spend/revenue/conversion values distort KPIs.
essential = ["Date", "Season", "Channel", "Service Type", "Customer Type",
             "Ad Spend", "Conversions", "Revenue"]
before = len(clean_df)
clean_df = clean_df.dropna(subset=essential).copy()
missing_rows_removed = before - len(clean_df)

# Normalize category labels.
for col in ["Season", "Channel", "Service Type", "Customer Type"]:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Basic validity rules: negative spend, conversions or revenue are invalid
# for this dataset's stated business context. Keep zero values as valid outcomes.
invalid_mask = (
    (clean_df["Ad Spend"] < 0) |
    (clean_df["Conversions"] < 0) |
    (clean_df["Revenue"] < 0)
)
invalid_rows = int(invalid_mask.sum())
clean_df = clean_df.loc[~invalid_mask].copy()

clean_df = clean_df.sort_values("Date").reset_index(drop=True)

print(f"Original rows: {len(raw_df):,}")
print(f"Exact duplicates removed: {duplicates_removed:,}")
print(f"Rows removed for missing essential values: {missing_rows_removed:,}")
print(f"Rows removed for negative metrics: {invalid_rows:,}")
print(f"Final clean rows: {len(clean_df):,}")
display(clean_df.head())

In [ ]:
# Re-check data quality after cleaning.
post_clean_quality = pd.DataFrame({
    "missing": clean_df.isna().sum(),
    "dtype": clean_df.dtypes.astype(str),
    "unique": clean_df.nunique()
})
display(post_clean_quality)

## 6. Feature Engineering — Business & Marketing KPIs

In [ ]:
analysis_df = clean_df.copy()

analysis_df["Year"] = analysis_df["Date"].dt.year
analysis_df["Month"] = analysis_df["Date"].dt.month
analysis_df["Month_Name"] = analysis_df["Date"].dt.strftime("%b")
analysis_df["Year_Month"] = analysis_df["Date"].dt.to_period("M").astype(str)
analysis_df["Quarter"] = analysis_df["Date"].dt.to_period("Q").astype(str)
analysis_df["Day_Name"] = analysis_df["Date"].dt.day_name()
analysis_df["Day_of_Week"] = analysis_df["Date"].dt.dayofweek

# Marketing metrics. ROAS is undefined when ad spend is zero.
analysis_df["ROAS"] = np.where(
    analysis_df["Ad Spend"] > 0,
    analysis_df["Revenue"] / analysis_df["Ad Spend"],
    np.nan
)
analysis_df["CPA"] = np.where(
    analysis_df["Conversions"] > 0,
    analysis_df["Ad Spend"] / analysis_df["Conversions"],
    np.nan
)
analysis_df["Revenue_per_Conversion"] = np.where(
    analysis_df["Conversions"] > 0,
    analysis_df["Revenue"] / analysis_df["Conversions"],
    np.nan
)
analysis_df["Revenue_per_Record"] = analysis_df["Revenue"]
analysis_df["Spend_per_Record"] = analysis_df["Ad Spend"]

# Record-level ratio is not the same as a true user/session conversion rate.
# No click, impression, reach, or traffic denominator exists in the dataset.
analysis_df["Revenue_to_Spend_Difference"] = (
    analysis_df["Revenue"] - analysis_df["Ad Spend"]
)

display(analysis_df.head())

In [ ]:
total_revenue = analysis_df["Revenue"].sum()
total_spend = analysis_df["Ad Spend"].sum()
total_conversions = analysis_df["Conversions"].sum()
overall_roas = total_revenue / total_spend if total_spend else np.nan
overall_cpa = total_spend / total_conversions if total_conversions else np.nan
revenue_per_conversion = total_revenue / total_conversions if total_conversions else np.nan

kpi_summary = pd.DataFrame({
    "Metric": ["Records", "Total Revenue", "Total Ad Spend", "Total Conversions",
               "Overall ROAS", "Overall CPA", "Revenue per Conversion"],
    "Value": [len(analysis_df), total_revenue, total_spend, total_conversions,
              overall_roas, overall_cpa, revenue_per_conversion]
})
display(kpi_summary)

## 7. Exploratory Data Analysis — Univariate & Bivariate

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.histplot(analysis_df["Revenue"], kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Revenue Distribution")
sns.histplot(analysis_df["Ad Spend"], kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Advertising Spend Distribution")
sns.countplot(data=analysis_df, x="Customer Type", ax=axes[1, 0])
axes[1, 0].set_title("Records by Customer Type")
sns.countplot(data=analysis_df, x="Service Type",
              order=analysis_df["Service Type"].value_counts().index, ax=axes[1, 1])
axes[1, 1].set_title("Records by Service Type")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=analysis_df, x="Channel", y="Revenue", ax=axes[0])
axes[0].tick_params(axis="x", rotation=35)
axes[0].set_title("Revenue by Channel")
sns.boxplot(data=analysis_df, x="Service Type", y="Revenue", ax=axes[1])
axes[1].tick_params(axis="x", rotation=25)
axes[1].set_title("Revenue by Service Type")
sns.boxplot(data=analysis_df, x="Customer Type", y="Revenue", ax=axes[2])
axes[2].set_title("Revenue by Customer Type")
plt.tight_layout()
plt.show()

In [ ]:
# Category-level summary helper: ratios are calculated from summed totals,
# not by averaging row-level ratios.
def grouped_performance(data, group_cols):
    out = data.groupby(group_cols, dropna=False).agg(
        Records=("Revenue", "size"),
        Revenue=("Revenue", "sum"),
        Ad_Spend=("Ad Spend", "sum"),
        Conversions=("Conversions", "sum"),
        Avg_Revenue_per_Record=("Revenue", "mean"),
    ).reset_index()
    out["Revenue_Share_pct"] = out["Revenue"] / out["Revenue"].sum() * 100
    out["Spend_Share_pct"] = out["Ad_Spend"] / out["Ad_Spend"].sum() * 100
    out["ROAS"] = np.where(out["Ad_Spend"] > 0, out["Revenue"] / out["Ad_Spend"], np.nan)
    out["CPA"] = np.where(out["Conversions"] > 0, out["Ad_Spend"] / out["Conversions"], np.nan)
    out["Revenue_per_Conversion"] = np.where(
        out["Conversions"] > 0, out["Revenue"] / out["Conversions"], np.nan
    )
    return out.sort_values("Revenue", ascending=False)

display(grouped_performance(analysis_df, ["Channel"]))
display(grouped_performance(analysis_df, ["Service Type"]))

## 8. Customer Analytics — New vs Returning

In [ ]:
customer_summary = grouped_performance(analysis_df, ["Customer Type"])
customer_summary["Conversion_Share_pct"] = (
    customer_summary["Conversions"] / customer_summary["Conversions"].sum() * 100
)
display(customer_summary)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
sns.barplot(data=customer_summary, x="Customer Type", y="Revenue", ax=axes[0])
axes[0].set_title("Revenue by Customer Type")
sns.barplot(data=customer_summary, x="Customer Type", y="Conversions", ax=axes[1])
axes[1].set_title("Conversions by Customer Type")
sns.barplot(data=customer_summary, x="Customer Type", y="ROAS", ax=axes[2])
axes[2].set_title("ROAS by Customer Type")
plt.tight_layout()
plt.show()

In [ ]:
# Revenue contribution and customer-type mix.
customer_contribution = analysis_df.groupby("Customer Type").agg(
    Revenue=("Revenue", "sum"),
    Ad_Spend=("Ad Spend", "sum"),
    Conversions=("Conversions", "sum"),
    Records=("Customer Type", "size")
)
customer_contribution["Revenue_Contribution_pct"] = (
    customer_contribution["Revenue"] / total_revenue * 100 if total_revenue else 0
)
customer_contribution["Record_Share_pct"] = (
    customer_contribution["Records"] / len(analysis_df) * 100
)
display(customer_contribution.sort_values("Revenue", ascending=False))

## 9. Sales & Marketing Analytics — Revenue, Services, Channels & Seasons

In [ ]:
channel_summary = grouped_performance(analysis_df, ["Channel"])
service_summary = grouped_performance(analysis_df, ["Service Type"])
season_summary = grouped_performance(analysis_df, ["Season"])

print("CHANNEL PERFORMANCE")
display(channel_summary)
print("SERVICE PERFORMANCE")
display(service_summary)
print("SEASON PERFORMANCE")
display(season_summary)

In [ ]:
fig = px.bar(channel_summary, x="Channel", y="Revenue", color="ROAS",
             title="Channel Revenue and ROAS", text_auto=".2s",
             color_continuous_scale="Blues")
fig.show()

fig = px.bar(service_summary, x="Service Type", y="Revenue", color="ROAS",
             title="Service Type Revenue and ROAS", text_auto=".2s",
             color_continuous_scale="Teal")
fig.show()

fig = px.bar(season_summary, x="Season", y="Revenue", color="ROAS",
             title="Seasonal Revenue and ROAS", text_auto=".2s")
fig.show()

In [ ]:
# Channel x service type performance matrix.
channel_service = grouped_performance(analysis_df, ["Channel", "Service Type"])
display(channel_service.sort_values(["ROAS", "Revenue"], ascending=False))
pivot_revenue = analysis_df.pivot_table(
    index="Channel", columns="Service Type", values="Revenue", aggfunc="sum", fill_value=0
)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_revenue, annot=True, fmt=",.0f", cmap="YlGnBu")
plt.title("Revenue: Channel x Service Type")
plt.tight_layout()
plt.show()

In [ ]:
# Customer type x channel and customer type x service comparisons.
display(grouped_performance(analysis_df, ["Customer Type", "Channel"]))
display(grouped_performance(analysis_df, ["Customer Type", "Service Type"]))

## 10. Segmentation — Available Segmentations & RFM Feasibility

In [ ]:
# Aggregate performance segmentation: classify channel/service combinations
# using observed ROAS and revenue. Thresholds use the dataset median to avoid
# imposing arbitrary external benchmarks.
segment_base = grouped_performance(analysis_df, ["Channel", "Service Type"])
roas_median = segment_base["ROAS"].median()
revenue_median = segment_base["Revenue"].median()

def performance_segment(row):
    if pd.isna(row["ROAS"]):
        return "Insufficient spend data"
    high_roas = row["ROAS"] >= roas_median
    high_revenue = row["Revenue"] >= revenue_median
    if high_roas and high_revenue:
        return "Scale candidate"
    if high_roas and not high_revenue:
        return "Test for growth"
    if not high_roas and high_revenue:
        return "Optimize efficiency"
    return "Review / reduce"

segment_base["Performance Segment"] = segment_base.apply(performance_segment, axis=1)
display(segment_base.sort_values(["Performance Segment", "Revenue"], ascending=[True, False]))

In [ ]:
# RFM assessment: the available table is campaign/performance-level, not
# transaction-level customer data. Customer Type is not a unique customer key.
rfm_requirements = pd.DataFrame({
    "RFM Component": ["Recency", "Frequency", "Monetary"],
    "Required Data": [
        "Customer ID + transaction date + transaction/order ID",
        "Customer ID + unique transaction/order ID",
        "Customer ID + transaction-level revenue"
    ],
    "Available in current dataset?": ["No", "No", "No"],
    "Reason": [
        "Date exists, but no customer identifier links a date to a person",
        "No customer identifier or order identifier",
        "Revenue exists only per record; no customer-level grouping key"
    ]
})
display(rfm_requirements)
print("Conclusion: true RFM scores and customer-level RFM segments cannot be calculated from this file.")

## 11. Loyalty Analysis — Retention, Repeat Purchases & Loyalty Opportunities

In [ ]:
# What can be measured: share of records/revenue tagged Returning.
# What cannot be measured: customer retention rate, repeat purchase rate,
# churn, cohort retention, and time between purchases without Customer ID/order history.
loyalty_proxy = customer_summary[[
    "Customer Type", "Records", "Revenue", "Conversions", "ROAS",
    "Revenue_Share_pct"
]].copy()
loyalty_proxy["Interpretation"] = loyalty_proxy["Customer Type"].map({
    "Returning": "Returning-labelled record contribution (not verified customer retention)",
    "New": "New-labelled record contribution"
})
display(loyalty_proxy)

loyalty_data_requirements = pd.DataFrame({
    "Metric": ["Repeat Purchase Rate", "Retention Rate", "Cohort Retention",
               "Churn Rate", "Purchase Interval"],
    "Missing fields": [
        "Customer ID, Order ID",
        "Customer ID, first and subsequent transaction dates",
        "Customer ID, transaction date",
        "Customer ID, activity window / last transaction date",
        "Customer ID, transaction timestamps"
    ]
})
display(loyalty_data_requirements)

### Loyalty opportunity framework
- Compare the revenue and ROAS contribution of records labelled **Returning** with **New**.
- If returning-labelled records contribute meaningful revenue, test VIP benefits and early access.
- Build a customer-level purchase history before claiming retention improvement.
- Track campaign cohorts and repeat transactions after a loyalty campaign is launched.

## 12. Upselling & Cross-selling — What the Dataset Supports

In [ ]:
# Service-type revenue and revenue per conversion can indicate commercial
# opportunities, but cannot prove that one customer upgraded between services.
service_opportunities = service_summary.copy()
service_opportunities["Revenue_per_Conversion"] = (
    service_opportunities["Revenue"] / service_opportunities["Conversions"].replace(0, np.nan)
)
display(service_opportunities[[
    "Service Type", "Records", "Revenue", "Revenue_Share_pct",
    "Conversions", "ROAS", "CPA", "Revenue_per_Conversion"
]])

# Product affinity requires item-level baskets (order_id, product_id).
affinity_requirements = pd.DataFrame({
    "Required field": ["Order ID", "Product ID", "Quantity", "Unit Price"],
    "Purpose": [
        "Identify items purchased in the same basket",
        "Identify product pairs and categories",
        "Measure item volume",
        "Calculate item-level sales and basket value"
    ],
    "Available?": ["No", "No", "No", "No"]
})
display(affinity_requirements)
print("Product affinity / market-basket rules are not run because item-level basket data is absent.")

## 13. Recommendation Engine — Rule-Based Business Recommendations

In [ ]:
recommendations = []

# Channel rules use aggregate ROAS and revenue contribution.
for _, row in channel_summary.iterrows():
    if pd.notna(row["ROAS"]) and row["ROAS"] >= channel_summary["ROAS"].median():
        recommendations.append({
            "Area": "Marketing Channel",
            "Finding": f"{row['Channel']} has ROAS {row['ROAS']:.2f}.",
            "Recommended Action": "Protect the channel's budget and run a controlled scale-up test.",
            "Priority": "High" if row["Revenue"] >= channel_summary["Revenue"].median() else "Medium"
        })
    else:
        recommendations.append({
            "Area": "Marketing Channel",
            "Finding": f"{row['Channel']} is below the median channel ROAS.",
            "Recommended Action": "Audit targeting, creative, offer and audience quality before increasing spend.",
            "Priority": "High"
        })

# Service rules.
best_service = service_summary.sort_values("ROAS", ascending=False).iloc[0]
recommendations.append({
    "Area": "Service Mix",
    "Finding": f"{best_service['Service Type']} has the highest observed service-level ROAS.",
    "Recommended Action": "Test this service in campaign creatives and relevant bundles; validate margin before scaling.",
    "Priority": "High"
})

# Customer-type rules.
if "Returning" in set(customer_summary["Customer Type"]):
    ret = customer_summary.loc[customer_summary["Customer Type"] == "Returning"].iloc[0]
    recommendations.append({
        "Area": "Loyalty",
        "Finding": f"Returning-labelled records account for {ret['Revenue_Share_pct']:.1f}% of recorded revenue.",
        "Recommended Action": "Pilot a returning-customer benefit and measure incremental revenue against a control group.",
        "Priority": "High"
    })

# Zero-revenue / spend audit.
zero_rev_spend = int(((analysis_df["Revenue"] == 0) & (analysis_df["Ad Spend"] > 0)).sum())
recommendations.append({
    "Area": "Campaign Efficiency",
    "Finding": f"{zero_rev_spend} records have positive ad spend and zero recorded revenue.",
    "Recommended Action": "Audit tracking, attribution windows, landing-page performance and campaign targeting for these records.",
    "Priority": "High" if zero_rev_spend > 0 else "Low"
})

recommendation_df = pd.DataFrame(recommendations)
display(recommendation_df)

In [ ]:
# Downloadable business recommendation output.
recommendation_df.to_csv("Moda_Lane_Business_Recommendations.csv", index=False)
print("Saved: Moda_Lane_Business_Recommendations.csv")

## 14. Advanced Analysis — Correlations, Trends & Patterns

In [ ]:
# Correlation is descriptive and does not establish causation.
numeric_for_corr = ["Ad Spend", "Conversions", "Revenue", "ROAS", "CPA",
                    "Revenue_per_Conversion"]
corr = analysis_df[numeric_for_corr].replace([np.inf, -np.inf], np.nan).corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Numeric KPI Correlation Matrix")
plt.tight_layout()
plt.show()
display(corr)

In [ ]:
# Monthly trends.
monthly = analysis_df.groupby("Year_Month").agg(
    Revenue=("Revenue", "sum"),
    Ad_Spend=("Ad Spend", "sum"),
    Conversions=("Conversions", "sum"),
    Records=("Revenue", "size")
).reset_index()
monthly["ROAS"] = monthly["Revenue"] / monthly["Ad_Spend"].replace(0, np.nan)
monthly["CPA"] = monthly["Ad_Spend"] / monthly["Conversions"].replace(0, np.nan)

fig = px.line(monthly, x="Year_Month", y=["Revenue", "Ad_Spend"],
              markers=True, title="Monthly Revenue vs Advertising Spend")
fig.show()
fig = px.line(monthly, x="Year_Month", y="ROAS", markers=True,
              title="Monthly ROAS Trend")
fig.show()
display(monthly)

In [ ]:
# Weekday patterns (interpret carefully; dataset records may not represent store visits).
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = analysis_df.groupby("Day_Name").agg(
    Revenue=("Revenue", "sum"),
    Ad_Spend=("Ad Spend", "sum"),
    Conversions=("Conversions", "sum"),
    Records=("Revenue", "size")
).reindex(weekday_order).reset_index()
weekday["ROAS"] = weekday["Revenue"] / weekday["Ad_Spend"].replace(0, np.nan)
display(weekday)
fig = px.bar(weekday, x="Day_Name", y="Revenue", color="ROAS",
             title="Revenue by Day of Week")
fig.show()

In [ ]:
# Optional inferential analysis: one-way ANOVA across channels for revenue.
# This tests differences in record-level revenue distributions, not causal effects.
try:
    from scipy.stats import f_oneway, kruskal
    groups = [g["Revenue"].dropna().values
              for _, g in analysis_df.groupby("Channel") if len(g) > 1]
    if len(groups) >= 2:
        f_stat, p_value = f_oneway(*groups)
        h_stat, p_kw = kruskal(*groups)
        print(f"ANOVA F={f_stat:.3f}, p={p_value:.5f}")
        print(f"Kruskal-Wallis H={h_stat:.3f}, p={p_kw:.5f}")
        print("Interpretation: statistical difference is not proof that channel choice caused the difference.")
except Exception as e:
    print("Statistical test skipped:", e)

In [ ]:
# Optional simple linear relationship diagnostic (not a forecasting model).
# No train/test model is claimed because the dataset has no campaign controls
# or sufficiently defined prediction target/decision context.
if analysis_df[["Ad Spend", "Revenue"]].dropna().shape[0] > 2:
    corr_spend_revenue = analysis_df[["Ad Spend", "Revenue"]].corr().iloc[0, 1]
    print(f"Pearson correlation between Ad Spend and Revenue: {corr_spend_revenue:.3f}")
    print("Correlation alone cannot establish incremental return from additional spend.")

## 15. Business Insights — Automatically Generated Findings

In [ ]:
insights = []

top_channel = channel_summary.sort_values("Revenue", ascending=False).iloc[0]
best_roas_channel = channel_summary.sort_values("ROAS", ascending=False).iloc[0]
top_service_revenue = service_summary.sort_values("Revenue", ascending=False).iloc[0]
best_roas_service = service_summary.sort_values("ROAS", ascending=False).iloc[0]
top_season = season_summary.sort_values("Revenue", ascending=False).iloc[0]

insights.append(f"**Largest revenue channel:** {top_channel['Channel']} generated {top_channel['Revenue']:,.2f} ({top_channel['Revenue_Share_pct']:.1f}% of revenue).")
insights.append(f"**Highest channel ROAS:** {best_roas_channel['Channel']} achieved {best_roas_channel['ROAS']:.2f}x ROAS.")
insights.append(f"**Largest service revenue:** {top_service_revenue['Service Type']} generated {top_service_revenue['Revenue']:,.2f}.")
insights.append(f"**Highest service ROAS:** {best_roas_service['Service Type']} achieved {best_roas_service['ROAS']:.2f}x ROAS.")
insights.append(f"**Top season by revenue:** {top_season['Season']} generated {top_season['Revenue']:,.2f}.")
insights.append(f"**Overall ROAS:** {overall_roas:.2f}x." if pd.notna(overall_roas) else "**Overall ROAS:** unavailable because total spend is zero.")
insights.append(f"**Zero-revenue records with positive spend:** {zero_rev_spend}.")

for item in insights:
    display(Markdown("- " + item))

## 16. Recommendations — Actionable Business Strategy

### A. Marketing budget optimization
- Prioritize controlled experiments in channels with stronger aggregate ROAS.
- Do not allocate all budget to the highest ROAS channel automatically; check volume, attribution, margins, and statistical stability.
- Investigate positive-spend / zero-revenue records and validate tracking before changing budgets.

### B. Service and offer strategy
- Feature high-ROAS services in relevant campaigns.
- Test Premium / Standard / Discounted bundles, while evaluating gross margin and discount cost.
- Avoid interpreting revenue per conversion as profit or customer willingness to pay.

### C. Loyalty program
- Use returning-customer-labelled performance as an initial signal, not a retention rate.
- Test VIP benefits, birthday offers, early access, and targeted reactivation.
- Measure incremental lift with a holdout/control group.

### D. Upselling and cross-selling
- Test a premium upgrade path and complementary product bundles.
- Capture order-level product data to identify actual frequently co-purchased products.
- Compare attach rate, average basket value, and gross margin before/after the experiment.

### E. Measurement improvement
- Add customer_id, order_id, product_id, quantity, unit_price, discount, cost, gross_margin, campaign_id, impressions, clicks, and sessions.
- Maintain consistent campaign attribution windows and channel naming.
- Build a monthly KPI review and experimentation log.

In [ ]:
# Prioritized action plan based on measurable findings.
action_plan = pd.DataFrame([
    ["1", "Audit tracking", "Positive spend with zero revenue", "Validate conversion events and attribution", "Marketing / Analytics"],
    ["2", "Channel experiments", "Channel ROAS comparison", "Run controlled budget tests", "Marketing"],
    ["3", "Service offer tests", "Service-level revenue and ROAS", "Test service bundles and premium upgrades", "Commercial / Marketing"],
    ["4", "Loyalty pilot", "Returning-labelled revenue contribution", "Launch controlled loyalty experiment", "CRM / Marketing"],
    ["5", "Data enrichment", "No customer/order/product identifiers", "Add customer and transaction-level fields", "Data / Operations"],
], columns=["Priority", "Initiative", "Evidence", "Next Action", "Owner"])
display(action_plan)
action_plan.to_csv("Moda_Lane_Action_Plan.csv", index=False)

## 17. Conclusion — Final Business Report

In [ ]:
print("=" * 72)
print("MODA LANE BOUTIQUE — BUSINESS INTELLIGENCE SUMMARY")
print("=" * 72)
print(f"Records analyzed: {len(analysis_df):,}")
print(f"Date coverage: {analysis_df['Date'].min().date()} to {analysis_df['Date'].max().date()}")
print(f"Total revenue: {total_revenue:,.2f}")
print(f"Total advertising spend: {total_spend:,.2f}")
print(f"Total conversions: {total_conversions:,.0f}")
print(f"Overall ROAS: {overall_roas:.2f}x" if pd.notna(overall_roas) else "Overall ROAS: N/A")
print(f"Overall CPA: {overall_cpa:,.2f}" if pd.notna(overall_cpa) else "Overall CPA: N/A")
print("\\nTop revenue channel:", top_channel["Channel"])
print("Highest ROAS channel:", best_roas_channel["Channel"])
print("Top revenue service:", top_service_revenue["Service Type"])
print("\\nDATA LIMITATION:")
print("True RFM, individual CLV, retention/cohort analysis, repeat purchase rate,")
print("and product affinity require customer/order/product-level identifiers.")
print("=" * 72)

## Appendix A — Data Dictionary
| Field | Description | Analytical use |
|---|---|---|
| Date | Record date | Time trends, month, quarter, weekday |
| Season | Season label supplied in source | Seasonal comparisons |
| Channel | Marketing channel | Channel performance and ROAS |
| Service Type | Service/offer category | Service mix analysis |
| Customer Type | New or Returning label | Customer-type contribution |
| Ad Spend | Advertising spend for record | Spend, ROAS, CPA |
| Conversions | Recorded conversions | Conversion volume, CPA |
| Revenue | Recorded revenue | Revenue, contribution, ROAS |

## Appendix B — KPI Definitions
- **Revenue:** Sum of recorded revenue.
- **Ad Spend:** Sum of advertising spend.
- **ROAS:** Revenue / Ad Spend. This is not profit or ROI.
- **CPA:** Ad Spend / Conversions.
- **Revenue per Conversion:** Revenue / Conversions.
- **Revenue Contribution:** Group revenue / total revenue.
- **Record Share:** Group record count / total record count.

## Appendix C — Required Data for Advanced Analytics
| Analysis | Additional fields required |
|---|---|
| RFM | customer_id, order_id, transaction_date, transaction_revenue |
| Individual CLV | customer_id, transaction history, margin, acquisition cost, observation window |
| Retention / Cohorts | customer_id, first_purchase_date, transaction_date |
| Product affinity | order_id, product_id, quantity |
| Profitability | cost of goods, discount, operating costs or gross margin |
| Funnel conversion rate | impressions, clicks, sessions, visitors, conversions |

## Appendix D — Export Cleaned Data

In [ ]:
clean_df.to_csv("Moda_Lane_Cleaned_Data.csv", index=False)
analysis_df.to_csv("Moda_Lane_Analysis_Data.csv", index=False)
print("Exported:")
print("- Moda_Lane_Cleaned_Data.csv")
print("- Moda_Lane_Analysis_Data.csv")
print("- Moda_Lane_Business_Recommendations.csv")
print("- Moda_Lane_Action_Plan.csv")